In [1]:
from PIL import Image
import os
import json

In [2]:
data = json.load(open('./projects_data.json'))

In [3]:
def crop_to_aspect(img, aspect_ratio: float):
  width, height = img.size
  new_height = width / aspect_ratio

  if height <= new_height:
    return img, False  # skipped cropping

  return img.crop((0, 0, width, round(new_height))), True  # cropped image

In [7]:
def resize_width_height(img, new_width: int):
  old_width, old_height = img.size

  if old_width <= new_width:
    return img, False  # skip resizing

  new_height = round(new_width / old_width * old_height)
  return img.resize((new_width, new_height), Image.LANCZOS), True  # resized

In [8]:
img = Image.open('./joachim-lesne-couTAixLzNM-unsplash.jpg')

img, did_cropped = crop_to_aspect(img, 4/3)
did_cropped

img, did_resized = resize_width_height(img, 400)
did_resized

img.save("./example_processed.jpg")

True

True

In [14]:
def process_screenshots():
  aspect_ratio = 4/3
  max_width = 400

  for project in data:
    image_path = f'./assets/screenshots/{project['slug']}.png'
    if os.path.exists(image_path):
      break

  img = Image.open(image_path)

  # crop, resize

  img, cropped = crop_to_aspect(img, aspect_ratio)
  img, resized = resize_width_height(img, max_width)

  # convert to RGB (necessary for saving as JPG)
  converted = False
  if img.mode in ("RGBA", "LA"):
    img = img.convert("RGB")
    converted = True

  # save as JPG with slug name
  output_path = os.path.join(os.path.dirname(image_path), f'{project["slug"]}.jpg')
  img.save(output_path, "JPEG", quality=95)

  # status message
  status = []
  status.append("✅ Cropped" if cropped else "❌ Skipped cropping")
  status.append("✅ Resized" if resized else "❌ Skipped resizing")
  if converted:
    status.append("✅ Converted to RGB")

  print(f"Processed {project['slug']}.jpg ({', '.join(status)})")

In [15]:
process_screenshots()

Processed basic-calculator.jpg (✅ Cropped, ✅ Resized, ✅ Converted to RGB)
